# TP1 : déployer et valider le cluster KRaft

| Repère | Valeur |
|---|---|
| Moment | Jour 1, 11:05-11:45, après la démo du TP (11:00-11:05) |
| Compétences | C2 KRaft et quorum ; C3 Déployer, démarrer et valider un cluster |
| Slides de rappel | S18 (installation), S19 (`server.properties`), S24 (architecture du lab) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule |

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, les variables sont perdues : relancez la cellule d'outils, puis les cellules qui définissent les fonctions et les variables dont vous avez besoin. Ne relancez pas une cellule qui modifie le cluster (création d'un topic, arrêt d'un nœud) : elle échouerait, ou referait l'action.
- Chaque étape suit le même schéma : une explication, un **exemple en ligne de commande** à exécuter tel quel, une **zone `# TODO`** à compléter en Python, puis une **vérification** (une instruction `assert`) qui affiche `OK` ou s'arrête sur `AssertionError: À revoir : ...` avec la raison.
- Une ligne qui commence par `!` est envoyée au terminal, comme dans un terminal ouvert sur le poste. `!docker exec kafka-tools kafka-topics.sh ...` lance la commande dans le conteneur d'administration (`# [tools]` dans les énoncés Markdown) ; `!docker compose ...` s'exécute dans le dossier `lab/` (`# [hôte]`). Dans une ligne `!`, `{expression}` insère une valeur Python, et les accolades littérales se doublent : `{{{{.Name}}}}` donne `{{.Name}}` à Docker. `output = !command` range les lignes de la sortie dans une liste ; `output.n` les joint en un seul texte.
- Le client Python est `confluent-kafka` (la bibliothèque librdkafka). `AdminClient(CONF)` crée un client d'administration connecté au listener INTERNAL (`BS`). Le TP crée un client neuf par cellule : un client qui a déjà servi garde en cache le coordinateur des groupes, et échoue quand ce coordinateur change de broker.
- Chaque étape de la section 6 regroupe ses commandes, ses résultats attendus et sa vérification.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
KAFKA_UI = "http://kafka-ui:8080"
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Deux fonctions d'aide, écrites ici : elles ne font pas partie de confluent-kafka
def wait_for(condition, timeout=60, step=2, message="condition"):
    """Rappelle condition() jusqu'à obtenir une valeur vraie, et la renvoie."""
    start = time.time()
    while True:
        try:
            value = condition()
        except Exception:
            value = None
        if value:
            return value
        if time.time() - start >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(step)

def table(lines, headers):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in line) + " |" for line in lines]
    display(Markdown("\n".join(md)))

In [ ]:
import re
import time
from collections import Counter

import requests

print("Bootstrap INTERNAL :", BS)

## 1. Objectifs

À la fin du TP, vous savez :

- vérifier que les 3 nœuds du lab sont sains ;
- lire dans `server.properties` ce qui fait d'un nœud un broker et un contrôleur KRaft ;
- vérifier qu'un nœud a été formaté et à quel cluster il appartient ;
- identifier le contrôleur actif et l'état de réplication du quorum ;
- décrire le cluster avec le client Python, et savoir ce que ce client ne voit pas ;
- retrouver un événement dans les traces d'un nœud ;
- lire le journal de métadonnées `__cluster_metadata` ;
- constater qu'un redémarrage du contrôleur actif provoque une nouvelle élection sans interruption.

## 2. Prérequis

- Lab démarré dans un terminal, dans le dossier `lab/` : `scripts/start.sh extended` (ou `core` si votre poste a moins de 16 Go de RAM), puis `scripts/validate.sh`, qui doit afficher « Environnement valide. ».
- JupyterLab ouvert sur http://localhost:8888, avec ce notebook.
- Notions : slides S4 à S24 (topics, partitions, brokers, KRaft, listeners).

## 3. Architecture utilisée

```
                         Réseau Docker kafka-lab
  +----------------------------------------------------------------------+
  |   kafka-1 (az-a)          kafka-2 (az-b)          kafka-3 (az-c)     |
  |   broker + contrôleur     broker + contrôleur     broker + contrôleur |
  |        \                       |                       /             |
  |         +------ quorum KRaft : 3 votants (port 9093) -+              |
  |                                                                      |
  |   jupyter : ce notebook (client Python, docker compose)              |
  |   tools   : CLI Kafka (kafka-*.sh, $BS), appelée par docker exec    |
  |   kafka-ui : http://localhost:8080                                   |
  |   [extended] prometheus, grafana, kafka-exporter, loadgen            |
  +----------------------------------------------------------------------+
```

| Listener | Port | Protocole | Usage dans ce TP |
|---|---|---|---|
| CONTROLLER | 9093 | PLAINTEXT | Quorum KRaft, entre contrôleurs |
| INTERNAL | 19092 | PLAINTEXT | CLI de `tools` et client Python du notebook (`BS`) |
| EXTERNAL | 9092, 9192, 9292 | PLAINTEXT | Clients lancés sur l'hôte (non utilisé ici) |
| SECURE | 19094 | SASL_SSL | TP5 |

## 4. Contexte métier

NovaShop, site de commerce en ligne, migre ses flux de commandes, de paiements et de navigation vers Kafka 4.3. L'équipe plateforme, dont vous faites partie, reçoit le nouveau cluster de recette : 3 nœuds en mode KRaft, sans ZooKeeper.

Avant d'ouvrir le cluster aux équipes de développement, vous devez produire un procès-verbal de recette technique. Il doit prouver que le cluster est sain, que le quorum fonctionne, que les traces sont exploitables et qu'un redémarrage de nœud ne l'interrompt pas. Ce notebook le construit au fil des étapes : il s'affiche en fin de TP.

## 5. Énoncé

Collectez les preuves suivantes :

1. l'identifiant du cluster et le fait que chaque nœud a été formaté avec cet identifiant ;
2. l'identité du contrôleur actif et l'état de réplication du quorum ;
3. la liste des brokers enregistrés et la version de métadonnées en vigueur ;
4. la trace qui prouve l'élection du contrôleur actif ;
5. le comportement du cluster quand le contrôleur actif redémarre.

## 6. Étapes

### 6.1 Vérifier les conteneurs

Le lab a été démarré dans un terminal. On vérifie ici, depuis le notebook, que les 3 nœuds Kafka sont sains : leur healthcheck teste que le listener INTERNAL accepte les connexions.

**Exemple en ligne de commande** (équivalent de `docker compose ps` dans `lab/`) :

In [ ]:
!docker compose ps --format 'table {{{{.Name}}}}\t{{{{.Status}}}}'

In [ ]:
# TODO 6.1 : lister les nœuds Kafka sains
# La commande ci-dessous renvoie une ligne "nom etat_de_sante" par conteneur.
lines = !docker compose ps --format '{{{{.Name}}}} {{{{.Health}}}}' 2>&1

# Construisez la liste healthy_nodes : noms des conteneurs qui commencent par "kafka-"
# et dont l'état de santé vaut "healthy".
# Indice : "kafka-1 healthy".partition(" ") renvoie ("kafka-1", " ", "healthy").
healthy_nodes = []  # à compléter

print(healthy_nodes)

In [ ]:
# Vérification 6.1
assert sorted(healthy_nodes) == ["kafka-1", "kafka-2", "kafka-3"], "À revoir : 3 nœuds Kafka sains : %s" % healthy_nodes
print("OK  3 nœuds Kafka sains : %s" % healthy_nodes)

**Résultat attendu** : `kafka-1`, `kafka-2` et `kafka-3` en `Up ... (healthy)`, ainsi que `jupyter`, `kafka-tools` et `kafka-ui`. En mode `extended` s'ajoutent `prometheus`, `grafana`, `kafka-exporter`, `loadgen-producer` et `loadgen-consumer`.

### 6.2 Lire la configuration KRaft d'un nœud

Le fichier `server.properties` de chaque nœud est dans `lab/config/kafka/kafka-N/`. Il dit quel rôle joue le nœud (`process.roles`), son identifiant, comment joindre le quorum, où il écoute et où il range ses données.

**Exemple en ligne de commande** :

In [ ]:
!grep -E '^(process.roles|node.id|controller.quorum.bootstrap.servers|controller.listener.names|listeners|advertised.listeners|log.dirs)=' config/kafka/kafka-1/server.properties

In [ ]:
# TODO 6.2 : écrire la fonction properties(text, sep)
# Elle reçoit le contenu d'un fichier de propriétés et renvoie un dictionnaire {key: value}.
#   - ignorer les lignes vides et les commentaires (qui commencent par #) ;
#   - ignorer les lignes qui ne contiennent pas le séparateur ;
#   - couper chaque ligne au PREMIER séparateur seulement : une valeur peut contenir "=" ;
#   - retirer les espaces autour de la clé et de la valeur (méthode strip()).
def properties(text, sep="="):
    result = {}
    # à compléter
    return result


# Le dossier lab/ est monté dans le conteneur du notebook : lecture directe du fichier
with open("config/kafka/kafka-1/server.properties") as f:
    conf1 = properties(f.read())

print(conf1.get("process.roles"), "- node.id", conf1.get("node.id"))

In [ ]:
# Vérification 6.2
assert properties("a = 1=2\n# b=3\n\nc") == {"a": "1=2"}, "À revoir : properties() : commentaire, ligne vide et '=' dans la valeur"
print("OK  properties() : commentaire, ligne vide et '=' dans la valeur")
assert conf1.get("process.roles") == "broker,controller", "À revoir : kafka-1 est à la fois broker et contrôleur"
print("OK  kafka-1 est à la fois broker et contrôleur")
assert conf1.get("controller.quorum.bootstrap.servers", "").count(":9093") == 3, "À revoir : 3 adresses de contact du quorum"
print("OK  3 adresses de contact du quorum")

# Les 3 nœuds côte à côte
conf_lines = []
for n in (1, 2, 3):
    with open("config/kafka/kafka-%d/server.properties" % n) as f:
        c = properties(f.read())
    conf_lines.append((n, c.get("node.id"), c.get("process.roles"), c.get("broker.rack"), c.get("log.dirs")))
table(conf_lines, ["nœud", "node.id", "process.roles", "broker.rack", "log.dirs"])

**Résultat attendu** pour `kafka-1` :

```
process.roles=broker,controller
node.id=1
controller.quorum.bootstrap.servers=kafka-1:9093,kafka-2:9093,kafka-3:9093
controller.listener.names=CONTROLLER
listeners=CONTROLLER://:9093,INTERNAL://:19092,EXTERNAL://:9092,SECURE://:19094
advertised.listeners=CONTROLLER://kafka-1:9093,INTERNAL://kafka-1:19092,EXTERNAL://localhost:9092,SECURE://kafka-1:19094
log.dirs=/var/lib/kafka/data
```

### 6.3 Vérifier le formatage du stockage

En KRaft, un nœud ne démarre que si son répertoire de données a été formaté avec `kafka-storage.sh format`. Le formatage écrit `meta.properties`, qui fixe l'identité du nœud : `cluster.id`, `node.id` et `directory.id`.

**Exemple en ligne de commande** :

In [ ]:
!docker compose exec -T kafka-1 /opt/kafka/bin/kafka-storage.sh info -c /opt/lab/node/server.properties
!docker compose logs kafka-1 | grep -E 'start-node|Formatting|already formatted|Kafka Server started'

In [ ]:
# TODO 6.3 : relever le cluster.id des 3 nœuds
# Pour chaque nœud n de 1 à 3 :
#   - lisez le fichier /var/lib/kafka/data/meta.properties du conteneur kafka-n :
#       text = !docker compose exec -T kafka-{n} cat /var/lib/kafka/data/meta.properties
#   - transformez-le en dictionnaire avec properties() ;
#   - rangez son cluster.id dans le dictionnaire ids, avec n pour clé.
ids = {}

print(ids)

In [ ]:
# Vérification 6.3
assert len(ids) == 3, "À revoir : meta.properties lu sur les 3 nœuds"
print("OK  meta.properties lu sur les 3 nœuds")
assert len(set(ids.values())) == 1, "À revoir : un seul cluster.id pour les 3 nœuds : %s" % set(ids.values())
print("OK  un seul cluster.id pour les 3 nœuds : %s" % set(ids.values()))

**Résultat attendu** : `Found metadata: {cluster.id=W5FmqobtRp-BMcXlZhVKBw, directory.id=4EubdzSZTbykbeqlrbgz1Q, node.id=1, version=1}`. Au premier démarrage, les traces contiennent `Formatting dynamic metadata voter directory /var/lib/kafka/data with metadata.version 4.3-IV0.` ; aux démarrages suivants, `All of the log directories are already formatted.`

### 6.4 Interroger le quorum

Le quorum KRaft élit un leader parmi les contrôleurs votants : c'est le contrôleur actif, seul à écrire dans le journal de métadonnées. L'outil `kafka-metadata-quorum.sh` interroge le quorum ; le client Python n'a pas d'équivalent.

**Exemple en ligne de commande** :

In [ ]:
!docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --status
!docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --replication

In [ ]:
# TODO 6.4 : écrire la fonction quorum_state()
# Elle doit :
#   - lancer "kafka-metadata-quorum.sh --bootstrap-server $BS describe --status" dans une cellule : output = !docker exec kafka-tools ... ;
#   - joindre les lignes en un texte avec output.n, puis le transformer en dictionnaire avec properties(text, sep=":") ;
#   - renvoyer le couple (LeaderId, LeaderEpoch), convertis en entiers avec int().
def quorum_state():
    # à compléter
    return None, None


leader, epoch = quorum_state()
print("Contrôleur actif :", leader, "- époque :", epoch)

In [ ]:
# Vérification 6.4
assert leader in (1, 2, 3), "À revoir : contrôleur actif identifié : %s" % leader
print("OK  contrôleur actif identifié : %s" % leader)
assert isinstance(epoch, int) and epoch >= 1, "À revoir : époque du leader : %s" % epoch
print("OK  époque du leader : %s" % epoch)

**Résultat attendu** : `LeaderId` vaut 1, 2 ou 3 ; `LeaderEpoch` vaut 1 sur un lab qui vient de démarrer. `CurrentVoters` liste les 3 nœuds et `CurrentObservers` est vide. `describe --replication` affiche 1 ligne `Leader` et 2 lignes `Follower`, avec un `Lag` de 0 ou proche de 0.

### 6.5 Décrire le cluster en Python

Les applications et les outils d'administration voient le cluster à travers les brokers. Le client Python obtient l'identifiant du cluster et la liste des brokers par l'API `describe_cluster()`, équivalente de `kafka-cluster.sh` et `kafka-broker-api-versions.sh`.

**Exemple en ligne de commande** :

In [ ]:
!docker exec kafka-tools bash -c "kafka-broker-api-versions.sh --bootstrap-server $BS | grep '^kafka'"
!docker exec kafka-tools kafka-cluster.sh cluster-id --bootstrap-server $BS
!docker exec kafka-tools kafka-features.sh --bootstrap-server $BS describe

In [ ]:
# TODO 6.5 : décrire le cluster avec le client Python
# admin.describe_cluster() renvoie un "futur" : .result() attend la réponse du cluster.
# Le résultat a trois attributs : cluster_id, controller (un nœud) et nodes (liste de nœuds).
# Chaque nœud a les attributs id, host, port et rack.
#   1. Rangez le résultat dans la variable cluster.
#   2. Construisez brokers : liste de tuples (id, host, rack), triée par id (fonction sorted).
cluster = None
brokers = []  # à compléter

print(brokers)

In [ ]:
# Vérification 6.5
assert cluster is not None and cluster.cluster_id == ids.get(1), "À revoir : même identifiant de cluster qu'à l'étape 6.3"
print("OK  même identifiant de cluster qu'à l'étape 6.3")
assert [b[2] for b in brokers] == ["az-a", "az-b", "az-c"], "À revoir : 3 brokers, un par rack : %s" % brokers
print("OK  3 brokers, un par rack : %s" % brokers)

Le résultat de `describe_cluster()` contient aussi un champ `controller`. Comparez-le, sur 5 appels, au leader du quorum lu à l'étape 6.4. Qu'observez-vous ? Notez votre explication avant de lire la correction.

In [ ]:
admin = AdminClient(CONF)
for i in range(5):
    print("describe_cluster().controller :", admin.describe_cluster().result().controller.id,
          "| leader du quorum :", quorum_state()[0])

**Résultat attendu** : 3 brokers `kafka-1`, `kafka-2`, `kafka-3` dans les racks `az-a`, `az-b`, `az-c`. `metadata.version` est finalisée à `4.3-IV0` et `kraft.version` à 1.

### 6.6 Défi : retrouver l'élection du contrôleur dans les traces

Chaque nœud écrit ses traces dans `/opt/kafka/logs`. Le nœud qui devient contrôleur actif écrit la ligne `Becoming the active controller` dans `controller.log`.

**Exemple en ligne de commande** :

In [ ]:
!docker compose exec -T kafka-1 ls /opt/kafka/logs

In [ ]:
# TODO 6.6 (défi) : trouver le nœud qui porte la trace d'élection
# Pour chaque nœud n de 1 à 3, cherchez la ligne "Becoming the active controller"
# dans ses fichiers de traces du contrôleur, puis rangez dans élections[n]
# la liste des lignes trouvées (liste vide si aucune).
# Commande de départ pour kafka-1, incomplète : elle ne lit qu'un seul fichier.
#   docker compose exec -T kafka-1 grep -h "Becoming the active controller" /opt/kafka/logs/controller.log
# Indice : relisez la liste des fichiers ci-dessus, et pensez à l'endroit où l'étoile est développée.
elections = {}

for n, found in elections.items():
    print("kafka-%d : %d ligne(s)" % (n, len(found)))

In [ ]:
# Vérification 6.6
assert len(elections) == 3, "À revoir : traces des 3 nœuds examinées"
print("OK  traces des 3 nœuds examinées")
assert leader in [n for n, found in elections.items() if found], "À revoir : le contrôleur actif (%s) porte une trace d'élection" % leader
print("OK  le contrôleur actif (%s) porte une trace d'élection" % leader)

**Résultat attendu** : sur un lab qui vient de démarrer, une seule ligne, sur le nœud élu leader : `Becoming the active controller at epoch 1`. Un nœud qui a déjà été contrôleur actif après un redémarrage porte aussi des lignes, avec des époques croissantes.

### 6.7 Explorer le journal de métadonnées

En KRaft, les métadonnées (topics, partitions, brokers, configurations) sont un journal Kafka : la partition `__cluster_metadata-0`, répliquée entre les contrôleurs. `kafka-dump-log.sh --cluster-metadata-decoder` décode ses enregistrements.

**Exemple en ligne de commande** :

In [ ]:
!docker compose exec -T kafka-1 ls /var/lib/kafka/data/__cluster_metadata-0
# Nombre d'enregistrements par type
!docker compose exec -T kafka-1 sh -c '/opt/kafka/bin/kafka-dump-log.sh --cluster-metadata-decoder --files /var/lib/kafka/data/__cluster_metadata-0/00000000000000000000.log 2>/dev/null | grep -o "\"type\":\"[A-Z_]*\"" | sort | uniq -c | sort -rn'

In [ ]:
# TODO 6.7 : compter les types d'enregistrements en Python
# La sortie complète du décodage est longue : elle n'est pas affichée.
dump = !docker compose exec -T kafka-1 sh -c '/opt/kafka/bin/kafka-dump-log.sh --cluster-metadata-decoder --files /var/lib/kafka/data/__cluster_metadata-0/00000000000000000000.log 2>/dev/null' 2>&1
dump = dump.n

# 1. Trouvez tous les types avec re.findall(r'"type":"([A-Z_]+)"', dump),
#    comptez-les avec Counter et rangez le résultat dans types.
# 2. Rangez dans created_topics les noms des topics créés (enregistrements TOPIC_RECORD) :
#    re.findall(r'"type":"TOPIC_RECORD".*?"name":"([^"]+)"', dump)
types = Counter()
created_topics = []

print(types.most_common(5))
print(created_topics)

In [ ]:
# Vérification 6.7
assert "__consumer_offsets" in created_topics, "À revoir : création de __consumer_offsets retrouvée"
print("OK  création de __consumer_offsets retrouvée")
assert types["TOPIC_RECORD"] == len(created_topics), "À revoir : un TOPIC_RECORD par topic créé (%d)" % len(created_topics)
print("OK  un TOPIC_RECORD par topic créé (%d)" % len(created_topics))
assert types["PARTITION_RECORD"] >= 50, "À revoir : " + ("%d PARTITION_RECORD (50 pour __consumer_offsets, plus les autres topics)"
         % types["PARTITION_RECORD"])
print("OK  " + ("%d PARTITION_RECORD (50 pour __consumer_offsets, plus les autres topics)"
         % types["PARTITION_RECORD"]))

**Résultat attendu** : un segment `00000000000000000000.log`, un fichier `quorum-state` et un instantané `.checkpoint`. Les types les plus nombreux sont `NO_OP_RECORD` et `PARTITION_RECORD`. On trouve un `TOPIC_RECORD` par topic : `__consumer_offsets`, plus `loadgen` et `lab-validate` en mode `extended`.

### 6.8 Kafka UI et son API

Ouvrez http://localhost:8080, puis les menus « Brokers » et « Topics » : nombre de brokers, contrôleur, liste des topics. L'interface expose aussi une API HTTP, joignable depuis le notebook à l'adresse `KAFKA_UI` (`http://kafka-ui:8080`).

In [ ]:
# TODO 6.8 : interroger l'API de Kafka UI
# requests.get(KAFKA_UI + "/api/clusters", timeout=10).json() renvoie une liste :
# un élément (dictionnaire) par cluster déclaré dans l'interface.
# Rangez le premier élément dans ui.
ui = {}

print(ui.get("name"), ui.get("status"), ui.get("brokerCount"))

In [ ]:
# Vérification 6.8
assert ui.get("status") == "ONLINE", "À revoir : cluster %s ONLINE dans Kafka UI" % ui.get("name")
print("OK  cluster %s ONLINE dans Kafka UI" % ui.get("name"))
assert ui.get("brokerCount") == 3, "À revoir : 3 brokers vus par Kafka UI"
print("OK  3 brokers vus par Kafka UI")

### 6.9 Redémarrer le contrôleur actif

Un arrêt contrôlé du contrôleur actif déclenche une nouvelle élection parmi les votants restants. Le nœud redémarré rejoint ensuite le quorum comme follower. Le trafic `loadgen` continue pendant toute l'opération : regardez Grafana si le mode `extended` tourne.

**Équivalent en ligne de commande** (ne pas lancer en plus du TODO) :

```bash
# [hôte] remplacer N par le LeaderId
docker compose restart kafka-N
docker compose exec tools kafka-metadata-quorum.sh --bootstrap-server kafka-1:19092,kafka-2:19092,kafka-3:19092 describe --status
docker compose ps kafka-N
```

In [ ]:
# TODO 6.9 : redémarrer le contrôleur actif et mesurer la bascule
#   1. Relevez (before, epoch_before) avec quorum_state().
#   2. Notez l'heure de départ : start = time.time().
#   3. Redémarrez le nœud : `!docker compose restart kafka-{before}`.
#   4. Écrivez une fonction new_leader() qui renvoie quorum_state() si le leader
#      a changé, et None sinon. Passez-la à wait_for(new_leader, timeout=60)
#      pour obtenir (after, epoch_after).
#   5. Calculez duration = time.time() - début.
before, epoch_before = None, None
after, epoch_after = None, None
duration = None

print("Leader avant :", before, "- après :", after, "- durée : %s s" % duration)

In [ ]:
# Vérification 6.9
assert after not in (None, before), "À revoir : nouveau leader du quorum : %s (avant : %s)" % (after, before)
print("OK  nouveau leader du quorum : %s (avant : %s)" % (after, before))
assert epoch_after is not None and epoch_after > epoch_before, "À revoir : époque passée de %s à %s" % (epoch_before, epoch_after)
print("OK  époque passée de %s à %s" % (epoch_before, epoch_after))

# Retour du nœud redémarré : healthy en 15 à 20 s, puis follower sans retard
def health(n):
    output = !docker compose ps --format '{{{{.Health}}}}' kafka-{n} 2>&1
    return output.n


wait_for(lambda: health(before) == "healthy", timeout=120, message="kafka-%d healthy" % before)
!docker compose logs kafka-{before} --since 2m | grep -E 'start-node|already formatted|Kafka Server started'
!docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --replication

**Résultat attendu** :

- le redémarrage dure quelques secondes ;
- immédiatement après, `LeaderId` désigne un autre nœud et `LeaderEpoch` a augmenté de 1 ;
- le nœud redémarré revient `healthy` en 15 à 20 s ; ses traces affichent `All of the log directories are already formatted.` puis `Kafka Server started` ;
- `describe --replication` le montre de nouveau `Follower` avec un `Lag` de 0.

## Procès-verbal de recette

La cellule suivante rassemble les preuves collectées. Une ligne « à compléter » renvoie à l'étape dont le `TODO` n'est pas terminé.

In [ ]:
def value(v, default="à compléter"):
    return default if v in (None, {}, [], "") else v


features = !docker exec kafka-tools kafka-features.sh --bootstrap-server $BS describe 2>&1
features = features.n
metadata_version = re.search(r"metadata\.version.*?FinalizedVersionLevel:\s*(\S+)", features)

table([
    ("Identifiant du cluster", value(cluster.cluster_id if cluster else None), "describe_cluster(), 6.5"),
    ("Nœuds formatés avec cet identifiant", value("%d/3" % len(ids) if ids else None), "meta.properties, 6.3"),
    ("Contrôleur actif et époque", value("%s, époque %s" % (leader, epoch) if leader else None), "describe --status, 6.4"),
    ("Brokers enregistrés", value(", ".join("%s (%s)" % (b[1], b[2]) for b in brokers)), "describe_cluster(), 6.5"),
    ("metadata.version", metadata_version.group(1) if metadata_version else "?", "kafka-features.sh describe"),
    ("Nœud portant la trace d'élection", value(", ".join("kafka-%d" % n for n, t in elections.items() if t)), "controller.log*, 6.6"),
    ("Nouveau contrôleur après redémarrage",
     value("%s, époque %s, en %s s" % (after, epoch_after, duration) if after else None), "6.9"),
], ["Preuve", "Valeur relevée", "Source"])

## 7. Questions de réflexion

1. Pourquoi le quorum compte-t-il 3 votants plutôt que 2 ou 4 ? Combien de pannes de contrôleur chaque taille tolère-t-elle ?
2. Pourquoi `advertised.listeners` diffère-t-il de `listeners` ? Que se passerait-il si un nœud annonçait un nom d'hôte inexistant ?
3. Où la liste des topics est-elle stockée en mode KRaft ? Qui l'écrit ?
4. Au redémarrage du contrôleur actif, les producteurs et consommateurs sont-ils interrompus ? Pourquoi ?
5. Dans ce lab, chaque nœud est à la fois broker et contrôleur. Quels risques ce choix présente-t-il en production ?
6. Pourquoi le formatage du stockage est-il une étape explicite en KRaft ?
7. Pourquoi le champ `controller` de `describe_cluster()` ne suffit-il pas à superviser le contrôleur actif ?